Today's topics:
* `def`, parameters, `return`
* default arguments and keyword calling
* scope: what a function can see
* docstrings

> HW6: write and test a small function of your own

# From copy-paste to a function

Lecture 5: one cell per moment order, `i = 1`, `i = 2`, `i = 3` -- same two lines, only the
number changing.

```python
i = 1
mu_i = np.mean((X - mu) ** i, axis=0)
mu_bar_i = np.round(mu_i / sigma ** i, 2)
print(f'mu_{i} = {mu_bar_i}')
```

*A function names a piece of code you plan to reuse -- write the logic once, call it as many
times as you need.*

# Defining a function

In [1]:
def square(x):
    y = x ** 2
    return y

print(square(3))
print(square(-4.5))

9
20.25


`return` hands a value back to the caller. `print` only displays -- it hands back nothing.

In [2]:
def square_print(x):
    print(x ** 2)     # displays, gives back None

result = square_print(5)   # prints 25
print(result)               # prints None -- gotcha

25
None


Multiple parameters:

In [3]:
def rectangle_area(w, h):
    return w * h

print(rectangle_area(3, 2))          # positional: w=3, h=2
print(rectangle_area(h=2, w=3))      # keyword: order doesn't matter

6
6


A function argument can be an array or a DataFrame column, exactly like any other value.

# Default arguments

In [4]:
def rectangle_area(w, h=1):
    return w * h

print(rectangle_area(3, 2))   # 6
print(rectangle_area(3))      # 3, uses default h=1
print(rectangle_area(w=3, h=5))

6
3
15


Rule: defaults come *after* non-defaults in the `def` line.

```python
def rectangle_area(h=1, w):   # SyntaxError: non-default argument follows default argument
    return w * h
```

> **Read more:** [W3Schools — Python Functions](https://www.w3schools.com/python/python_functions.asp)

# Scope: what a function can see

*A variable created inside a function -- who else can see it?*

In [5]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
def add_one(n):
    total = n + 1
    return total

add_one(5)
print(total)   # NameError: total is not defined

NameError: name 'total' is not defined

`total` is **local** -- exists only while `add_one` runs.

The other direction is more dangerous: a function *reading* a global it never received.

In [6]:
sigma = 2.0

def bad_zscore(x):
    return x / sigma       # silently depends on a variable outside the function

print(bad_zscore(4.0))
sigma = 10.0
print(bad_zscore(4.0))     # same call, same arguments, different answer!

2.0
0.4


Same call, two answers -- no error, just a silently wrong one.

*Fix:* if a function needs a value, put it in the parameter list.

In [7]:
def zscore_fixed(x, sigma):
    return x / sigma

print(zscore_fixed(4.0, 2.0))
print(zscore_fixed(4.0, 10.0))

2.0
0.4


Now the two different answers are explained by the two arguments, not invisible state --
reproducible no matter what order you ran cells in.

> **Read more:** [W3Schools — Python Scope](https://www.w3schools.com/python/python_scope.asp)

### [Check your understanding]

* Write a function `is_high_purity(pct)` that returns `True` if `pct >= 99.9`
* Give it one parameter, no default
* Test it on `99.95` and `98.2`

# Docstrings

A note for future-you, living inside the function:

In [8]:
def rectangle_area(w, h=1):
    """
    Compute the area of a rectangle.

    Parameters
    ----------
    w : float
        Width.
    h : float, default 1
        Height.

    Returns
    -------
    float
        The area w * h.
    """
    return w * h

In [9]:
help(rectangle_area)

Help on function rectangle_area in module __main__:

rectangle_area(w, h=1)
    Compute the area of a rectangle.

    Parameters
    ----------
    w : float
        Width.
    h : float, default 1
        Height.

    Returns
    -------
    float
        The area w * h.



`help()` reads the docstring; Jupyter's `Shift+Tab` pulls up the same text while typing a call.
For HW6, a clear one-line summary is enough -- the full NumPy-style sections above are shown once
as "what it can look like."

# Refactoring the moments code

## Dataset: Elemental Properties

This dataset contains physical and electronic properties of chemical elements,
including atomic mass, ionization energies, vacancy formation energies,
bulk static energies, and crystal structures.

Note that some entries contain `NaN` (Not a Number) values where data was
unavailable. We'll need to handle these when performing calculations.

In [10]:
import os
import pandas as pd

_file = 'elements.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e
data

,Symbol,Bulk Static Energy (eV),Reference Energy (eV),Clean Cohesive,Cohesive Energy (eV),Cohesive Energy Per Atom (eV),Atomic Number,Mendeleev Number,Atomic Mass,STP Phase,Natural Crystal Structure,Vacancy Formation Energy (eV),"Vacancy Migration Energy, non-disp (eV)","Vacancy Migration Energy, disp (eV)",Experimental Hvac,Experimental Hmig,Activation Energy (eV),NEBimageVolume,Ionization Energies (eV)
0,Ac,-442.228190,-0.25,1.0,415.09,3.843385,89,48,227.0000,solid,fcc,1.259565,0.460300,0.454340,NaN,NaN,1.719865,4900.00,5.170000
1,Ag,-305.971040,-0.35,1.0,268.38,2.485000,47,71,107.8600,solid,fcc,0.683455,0.695180,0.695030,1.12,0.60,1.378635,1930.00,7.576230
2,Al,-403.993240,-0.23,1.0,379.35,3.512481,13,80,26.9815,solid,fcc,0.607072,0.581350,0.581650,0.68,0.68,1.188422,1779.29,5.985768
3,Ar,-7.404033,-0.05,1.0,1.71,0.015855,18,3,39.9480,gas,fcc,0.049976,0.020634,NaN,NaN,NaN,0.070610,4859.00,15.759610
4,Au,-353.846200,-0.28,1.0,323.28,2.993363,79,70,196.9700,solid,fcc,0.400096,0.548710,0.547920,0.89,0.84,0.948806,1951.89,9.225530
5,Ba,NaN,NaN,0.0,0.00,0.000000,56,14,137.3200,solid,bcc,1.085024,0.000000,NaN,NaN,NaN,1.085024,0.00,5.211664
6,Be,-395.309910,-0.01,1.0,394.12,3.649236,4,77,9.0120,solid,hcp,-0.062697,0.750210,0.750340,NaN,NaN,0.687513,851.00,9.322700
7,Ca,-216.424440,-0.13,1.0,202.14,1.871703,20,16,40.0780,solid,fcc,1.133600,0.470580,NaN,NaN,NaN,1.604180,4554.00,6.113160
8,Cd,NaN,NaN,0.0,0.00,0.000000,48,75,112.4110,solid,hcp,0.302081,0.000000,NaN,0.45,NaN,0.302081,0.00,8.993820
9,Ce,-640.664090,-1.27,1.0,503.70,4.663924,58,32,140.1160,solid,fcc,1.298365,0.544760,NaN,NaN,NaN,1.843125,2818.00,5.538700


Same elements table since Lecture 8. One numeric column, `NaN`s dropped, 1D array:

In [11]:
import numpy as np

x = data['Ionization Energies (eV)'].dropna().values
x[:5]

array([ 5.17    ,  7.57623 ,  5.985768, 15.75961 ,  9.22553 ])

*What actually changes between the four copy-pasted moment cells? Only `i`.*

In [12]:
def standardized_moment(x, order=1):
    """
    Compute the standardized central moment of an array (see the moments cells above).

    Parameters
    ----------
    x : array-like
        The data values.
    order : int, default 1
        Which moment to compute (1 -> ~0, 2 -> ~1, by construction).

    Returns
    -------
    float
        The standardized central moment, mu_i / sigma**i.
    """
    mu = np.mean(x)
    sigma = np.std(x)
    mu_i = np.mean((x - mu) ** order)
    return mu_i / sigma ** order

for order in range(1, 5):
    print(f'mu_{order} = {standardized_moment(x, order):.3f}')

mu_1 = -0.000
mu_2 = 1.000
mu_3 = 2.947
mu_4 = 12.312


`order=1` -> ~0, `order=2` -> ~1 by construction -- sanity check, not a bug.

Four cells became one function + a loop. Now it generalizes along a dimension copy-paste
couldn't: columns.

In [13]:
for col in ['Atomic Mass', 'Ionization Energies (eV)', 'Vacancy Formation Energy (eV)']:
    x = data[col].dropna().values
    print(col, [round(standardized_moment(x, i), 2) for i in range(1, 5)])

Atomic Mass [np.float64(0.0), np.float64(1.0), np.float64(0.22), np.float64(1.84)]
Ionization Energies (eV) [np.float64(-0.0), np.float64(1.0), np.float64(2.95), np.float64(12.31)]
Vacancy Formation Energy (eV) [np.float64(0.0), np.float64(1.0), np.float64(0.23), np.float64(1.89)]


Same function, three columns, zero changes.

### [Check your understanding]

* Write `zscore(x, value)` -- how many standard deviations `value` is from the mean of array `x`
* One-line docstring
* Test on `data['Cohesive Energy Per Atom (eV)']` for a couple of specific elements
  (`data.loc[data['Symbol'] == 'Fe', 'Cohesive Energy Per Atom (eV)']` or similar)

Next time: a `DataFrame` is data bundled with a big collection of functions (methods) attached
to it.